### RAG Pipelines: Data Ingestion to vector db Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

c:\Users\user\Desktop\rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## load all the pdf files in the directory
def process_all_pdfs(pdf_directory):
    """Process all pdf files in a directory"""
    documents_list = []
    pdf_dir = Path(pdf_directory)

    # find all pdf files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    print(f"find {len(pdf_files)} to process")

    for pdf_file in pdf_files:
        print(f"processing {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf' 
            documents_list.extend(documents)
            print(f"loaded {len(documents)}pages")
        except Exception as e:
            print(f"error: {e}")
    print(f"total documents loaded {len(documents_list)}")
    return documents_list

In [3]:
all_pdf_documents = process_all_pdfs("../data")

find 3 to process
processing ..\data\pdf\paper.pdf
loaded 11pages
processing ..\data\pdf\paper2.pdf
loaded 8pages
processing ..\data\pdf\paper3.pdf
loaded 12pages
total documents loaded 31


In [ ]:
#all_pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-03-05T09:43:57+01:00', 'author': 'agimeno', 'moddate': '2018-03-12T10:24:10-04:00', 'source': '..\\data\\pdf\\paper.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'paper.pdf', 'file_type': 'pdf'}, page_content="The EUROCALL Review, Volume 25, No. 2, September 2017 \n \n 18 \nResearch paper \n \nA look at advanced learners’ use of mobile devices for \nEnglish language study: Insights from interview data \nMariusz Kruk \nUniversity of Zielona Gora, Poland \n______________________________________________________________ \nmkruk @ uz.zgora.pl \n  \nAbstract \nThe paper discusses the results of a study which explored advanced learners of English \nengagement with their mobile devices to develop learning experiences that meet their \nneeds and goals as foreign language learners. The data were collected from 20 students \nby means of a semi -structured interv

: 

In [4]:
## text splitting get into chunks
def split_documents(documents,chunk_size = 1000, chunk_overlap = 200):
    """split documnts into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators=["\n","\n\n",""," "]
    )
    split_docs = text_splitter.split_documents(documents)
    print(F"split {len(documents)} documents into {len(split_docs)} chunks")
    if split_docs:
        print("\n example chunk")
        print(f"content: {split_docs[0].page_content[:200]}...")
        print(f"meta data: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)
#chunks

split 31 documents into 151 chunks

 example chunk
content: The EUROCALL Review, Volume 25, No. 2, September 2017 
 
 18 
Research paper 
 
A look at advanced learners’ use of mobile devices for 
English language study: Insights from interview data 
Mariusz Kr...
meta data: {'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2018-03-05T09:43:57+01:00', 'author': 'agimeno', 'moddate': '2018-03-12T10:24:10-04:00', 'source': '..\\data\\pdf\\paper.pdf', 'total_pages': 11, 'page': 0, 'page_label': '1', 'source_file': 'paper.pdf', 'file_type': 'pdf'}


### Embedding & VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Tuple, Any, Dict
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """handle document embedding generation usin SentenceTransformer"""
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        """
        initialize the embedding manager

        args:
        model_name: huggingFace model name for sentence embedding
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """load the SentenceTransformer model"""
        try:
            print(f"Loading Embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successdully:\nmodel dimensions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    def generate_embedding(self, texts: List[str])->np.ndarray:
        """
        generate emmbiding for a list of texts

        args: list of texts to embbed

        return: numpy array of embeddings with shape (len(texts), emb_dime)
        """
        if not self.model:
            raise ValueError("Model Not Loaded")
        print(f"generate embedding for {len(texts)}...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"generated embeddings with shape: {embeddings.shape}")
        return embeddings
    def get_embedding_dimension(self)->int:
        """get the embedding dimension of the model"""
        if not self.model:
            return ValueError("Model not loaded")
        return self.model.get_sentence_embedding_dimension()
    
## Initialize the embedding manager
embedding_amanger = EmbeddingManager()
embedding_amanger

Loading Embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 132.22it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successdully:
model dimensions: 384


### VectorStore

In [8]:
class VectorStore:
    """manage documents embedding in a ChromaDb vectore Store"""
    def __init__(self, collection_name:str="documents_pdf", persist_directory:str="../data/vector_store"):
        """ininialize the vector store
        
        args: 
            collection_name: the name of ChromaDb collection
            persist_directory: Directory to persist the vectore store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client & collection"""
        try:
            # create presistent ChromaDb Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # get or create chromaDb collection
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata = {"description":"document embedding for RAG"}
            )
            print(f"Vetor store initialized. collection: {self.collection_name}")
            print(f"Existing documet in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error inintializing vector store: {e}")


    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """"
        add documents and thier embedding to the Vector Store
        """

        if len(documents) != len(embeddings):
            raise ValueError("documents length must Match embedding length")
        print(f"length of documents to add: {len(documents) }")

        ## preare data fro ChromaDb
        ids = []
        metadatas = []
        documents_list = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents,embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # document content
            documents_list.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings,
                metadatas=metadatas,
                documents=documents_list
            )
            print(f"successfully add {len(documents)} documents to the vector store")
            print(f"total documents in collection {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore



Vetor store initialized. collection: documents_pdf
Existing documet in collection: 0


In [9]:
### convert texts to embeddings
texts = [doc.page_content for doc in chunks]

## generate embeddings
embeddings = embedding_amanger.generate_embedding(texts)

## store in the vector database
vectorstore.add_documents(chunks,embeddings)

generate embedding for 151...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Batches: 100%|██████████| 5/5 [01:01<00:00, 12.37s/it]


generated embeddings with shape: (151, 384)
length of documents to add: 151
successfully add 151 documents to the vector store
total documents in collection 151


### Retriever Pipeline from Vector store

In [ ]:
class RAGRetreiver:
    """handles query-based retrieval from the vector store"""
    def __init__(self, vectorstore:VectorStore, embedding_amanger: EmbeddingManager):
        self.vectorstore = vectorstore
        self.embedding_amanger = embedding_amanger

: 